# Dataset Runner — Persistent Kernel Stability Test

This notebook runs `run_worker` in-process for maximum eGPU stability.

In [ ]:
import importlib.util
import sys
import time
from pathlib import Path
import torch

_script_path = Path("../scripts/generate_offline_dataset.py").resolve()
_spec = importlib.util.spec_from_file_location("gen_dataset", _script_path)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

run_worker = _mod.run_worker
_init_hdf5 = _mod._init_hdf5
_count_samples = _mod._count_samples

print(f"Script loaded from: {_script_path}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
import argparse
OUT_PATH = Path("../data/offline_dataset_notebook.h5").resolve()
ARGS = argparse.Namespace(
    out=str(OUT_PATH),
    seed=42,
    device="cuda" if torch.cuda.is_available() else "cpu",
    samples_per_run=50,
    nx=512,
    ny=256,
    nz=64,
    materials="SS316L,IN718",
    viz=True,
    wsl_safe=True,
    mlflow=True,
    mlflow_experiment="lpbf-dataset-gen-diagnostics",
    no_diag_csv=False,
    orchestration_id="notebook",
    no_triton=False,
)
N_RUNS = 3
COOLDOWN_S = 5
APPEND = False

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from IPython.display import Image, display
from neural_pbf.pipelines.dataset import prepare_trajectory, save_path_preview
from neural_pbf.physics.material import MaterialConfig
from neural_pbf.core.config import SimulationConfig
from neural_pbf.utils.units import LengthUnit

_zoo = {
    "SS316L": MaterialConfig.ss316l_preset(),
    "IN718":  MaterialConfig.in718_preset(),
}
_sim_cfg = SimulationConfig(
    Lx=1.0, Ly=0.5, Lz=0.125,
    Nx=ARGS.nx, Ny=ARGS.ny, Nz=ARGS.nz,
    length_unit=LengthUnit.MILLIMETERS,
)

_preview_dir = OUT_PATH.parent
_plans = []
for _run_idx in range(N_RUNS):
    _plan = prepare_trajectory(ARGS, _run_idx, _zoo)
    _png_path = _preview_dir / f"path_run_{_run_idx:03d}.png"
    save_path_preview(_plan, _png_path, _sim_cfg.Lx_m, _sim_cfg.Ly_m)
    _plans.append(_plan)
    print(f"Run {_run_idx}: {_plan.mat_key}")
    display(Image(filename=str(_png_path)))

In [ ]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
if not APPEND or not OUT_PATH.exists():
    _init_hdf5(OUT_PATH, ARGS)
    print(f"HDF5 initialised: {OUT_PATH}")

In [ ]:
run_results = []
for run_idx in range(N_RUNS):
    ARGS.run_index = run_idx
    t_start = time.perf_counter()
    status = "ok"
    try:
        print(f"Starting run {run_idx + 1}/{N_RUNS}")
        run_worker(ARGS, plan=_plans[run_idx] if run_idx < len(_plans) else None)
    except Exception as exc:
        status = f"FAILED: {exc}"
        print(f"Run {run_idx} failed: {exc}")
    finally:
        duration = time.perf_counter() - t_start
        run_results.append((run_idx, status, duration))
        if run_idx < N_RUNS - 1: time.sleep(COOLDOWN_S)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
dfs = []
for i in range(N_RUNS):
    p = OUT_PATH.with_suffix(f'.run{i:03d}.diag.csv')
    if p.exists(): dfs.append(pd.read_csv(p))
if dfs:
    df = pd.concat(dfs)
    df.plot(x='step_idx', y='peak_temperature')
    plt.show()